# Taco Alley Complaint Structuring with LoRA-based SFT

 > LoRA based fine-tuning workflow for a small language model that converts incoming complaints with fields `customer_id`, `date`, `location`, `message` into processed complaint JSON by adding `category`, `sub_category`, and `tone_urgency`.

### Minimum viable hardware guidance

- For this notebook's full fine-tuning path, use a small base model and conservative batch settings.
- A practical 16GB VRAM target is a model in the ~1B to ~3B range with gradient checkpointing and accumulation.
- LoRA/QLoRA often reduce memory substantially, but those implementations are deferred to the later notebook.

<a href="https://colab.research.google.com/github/ned1313/Fine-tuning-and-Optimizing-Small-Language-Models/blob/main/notebooks/sft_lora_run.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

In [ ]:
# Optional: Colab-only dependency bootstrap.
# Local environments can skip this cell if dependencies are already installed.
import importlib.util
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    required = [
        "transformers>=4.46.0",
        "trl>=0.11.0",
        "datasets>=2.21.0",
        "accelerate>=0.34.0",
        "sentencepiece>=0.2.0",
        "matplotlib>=3.8.0",
    ]
    missing = []
    for req in required:
        module_name = req.split(">=")[0].replace("-", "_")
        if importlib.util.find_spec(module_name) is None:
            missing.append(req)
    if missing:
        print("Installing missing packages:", missing)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    else:
        print("Required packages already available.")
else:
    print("Colab dependency bootstrap skipped on local runtime.")

## Demonstration: Preparing datasets as training input

 > Load `datasets/taco_alley_customer_complaints.csv`, validate the required complaint schema, and build train/eval splits for supervised fine-tuning.

In [ ]:
import csv
import json
import math
import random
import statistics
import time
from collections import Counter
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

# Reproducibility for split and sampling.
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# 16GB-friendly default base model for full fine-tuning.
BASE_MODEL = "Qwen/Qwen3-1.7B"
MAX_SEQ_LENGTH = 384
TRAIN_EVAL_SPLIT = 0.15

# Incoming complaints arrive with these fields.
INPUT_FIELDS = ["customer_id", "date", "location", "message"]
# Processed complaints add these fields.
OUTPUT_FIELDS = ["category", "sub_category", "tone_urgency"]
REQUIRED_FIELDS = [*INPUT_FIELDS, *OUTPUT_FIELDS]

EVAL_SAMPLE_SIZE = 120


def resolve_project_root(start: Path) -> Path:
    current = start.resolve()
    for parent in [current, *current.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    return current


PROJECT_ROOT = resolve_project_root(Path.cwd())
DATASETS_DIR = PROJECT_ROOT / "datasets"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
TRAINING_DATA_PATH = DATASETS_DIR / "taco_alley_customer_complaints.csv"
if not TRAINING_DATA_PATH.exists():
    raise FileNotFoundError(f"Training dataset not found at: {TRAINING_DATA_PATH}")

output_dir = ARTIFACTS_DIR / "sft_lora_run"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Training data source: {TRAINING_DATA_PATH}")

In [ ]:
import re

with TRAINING_DATA_PATH.open("r", encoding="utf-8", newline="") as f:
    raw_rows = list(csv.DictReader(f))

all_columns = list(raw_rows[0].keys())
missing_columns = [c for c in REQUIRED_FIELDS if c not in all_columns]
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

tone_allowed = {"low", "moderate", "high", "urgent"}
clean_examples = []
dropped_rows = 0

for row in raw_rows:
    record = {k: str(row.get(k, "")).strip() for k in REQUIRED_FIELDS}
    # Drop any rows that are missing required fields
    if any(not record[k] for k in REQUIRED_FIELDS):
        dropped_rows += 1
        continue
    # Drop rows with invalid customer_id matching CUST-{5digit number}
    if not re.match(r"^CUST-\d{5,}",record["customer_id"]):
        dropped_rows += 1
        continue
    # Drop rows with invalid tone_urgency values
    if record["tone_urgency"].lower() not in tone_allowed:
        dropped_rows += 1
        continue
    # Drop rows with invalid dates or dates in the future
    if not re.match(r"^\d{4}-\d{2}-\d{2}$", record["date"]):
        dropped_rows += 1
        continue
    if record["date"] > time.strftime("%Y-%m-%d"):
        dropped_rows += 1
        continue

    input_obj = {k: record[k] for k in INPUT_FIELDS}
    target_obj = {k: record[k] for k in REQUIRED_FIELDS}
    clean_examples.append(
        {
            "input_obj": input_obj,
            "target_obj": target_obj,
            "target_json": json.dumps(target_obj, ensure_ascii=False),
        }
    )

dataset = Dataset.from_list(clean_examples).train_test_split(test_size=TRAIN_EVAL_SPLIT, seed=SEED)
train_ds = dataset["train"]
eval_ds = dataset["test"]

category_counts = Counter(x["target_obj"]["category"] for x in clean_examples)
tone_counts = Counter(x["target_obj"]["tone_urgency"] for x in clean_examples)

print(f"Loaded rows: {len(raw_rows)}")
print(f"Rows kept: {len(clean_examples)}")
print(f"Rows dropped: {dropped_rows}")
print(f"Train size: {len(train_ds)}")
print(f"Eval size: {len(eval_ds)}")
print("Categories:", dict(category_counts))
print("Tone distribution:", dict(tone_counts))

In [ ]:
train_ds[0]

## Demonstration: Tokenizers and chat templates for structured complaints

> Convert each complaint pair into a chat-style training sample with a strict JSON target and inspect tokenization behavior before training.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer: {tokenizer.__class__.__name__}")
print(f"Tokenizer pad token: {tokenizer.pad_token}")
print(f"Tokenizer eos token: {tokenizer.eos_token}")

In [ ]:

SYSTEM_PROMPT = (
    "You are a complaint structuring assistant for Taco Alley. "
    "Input is a JSON object with fields customer_id, date, location, message. "
    "Return only valid JSON with exactly these fields in the output: "
    f"{REQUIRED_FIELDS}. "
    "Keep the original input fields unchanged and add correct values for category, sub_category, and tone_urgency. "
    "Do not add any extra keys or commentary."
)

def format_split_for_sft(split_ds: Dataset) -> Dataset:
    formatted_rows = []
    for ex in split_ds:
        train_messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps(ex["input_obj"], ensure_ascii=False)},
            {"role": "assistant", "content": ex["target_json"]},
        ]
        infer_messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps(ex["input_obj"], ensure_ascii=False)},
        ]

        try:
            text = tokenizer.apply_chat_template(
                train_messages, tokenize=False, add_generation_prompt=False, enable_thinking=False
            )
            prompt_text = tokenizer.apply_chat_template(
                infer_messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
            )
        except Exception:
            text = (
                f"SYSTEM: {SYSTEM_PROMPT}\n\n"
                f"USER: {json.dumps(ex['input_obj'], ensure_ascii=False)}\n\n"
                f"ASSISTANT: {ex['target_json']}"
            )
            prompt_text = (
                f"SYSTEM: {SYSTEM_PROMPT}\n\n"
                f"USER: {json.dumps(ex['input_obj'], ensure_ascii=False)}\n\n"
                "ASSISTANT:"
            )

        formatted_rows.append(
            {
                "input_obj": ex["input_obj"],
                "target_obj": ex["target_obj"],
                "target_json": ex["target_json"],
                "text": text,
                "prompt_text": prompt_text,
            }
        )
    return Dataset.from_list(formatted_rows)

train_ds = format_split_for_sft(train_ds)
eval_ds = format_split_for_sft(eval_ds)

train_lengths = [len(tokenizer(x["text"]).input_ids) for x in train_ds]
print(f"Sample train token length mean: {statistics.mean(train_lengths):.1f}")
print(f"Sample train token length p95: {sorted(train_lengths)[int(0.95 * (len(train_lengths)-1))]}")
print(f"Sample train token length max: {max(train_lengths)}")
print("\nFormatted sample:\n")
print(train_ds[0]["text"][:1200])

## Demonstration: Adding LoRA Configuration

> Build LoRAConfig for use with SFT

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    use_rslora=True,
    init_lora_weights="pissa",
)

## Demonstration: Configuring SFT training properties with TRL

> Build `SFTConfig` and `SFTTrainer` for a full fine-tuning run tuned for 16GB-class VRAM.

In [ ]:
from trl import SFTConfig
from trl import SFTTrainer

# Check to see if this is an AMD card
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
device_map = "auto" if torch.cuda.is_available() else None
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=dtype if torch.cuda.is_available() else torch.float32,
    use_safetensors=True,
    device_map=device_map,
)

model.config.use_cache = False  # Better memory behavior with gradient checkpointing.
model.gradient_checkpointing_enable()
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:

sft_config = SFTConfig(
    max_length=MAX_SEQ_LENGTH,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=10,
    seed=SEED,
    eval_strategy="steps",
    eval_steps=5,
    save_strategy="steps",
    save_steps=5,
    output_dir=str(output_dir / "checkpoints"),
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.03,
    max_grad_norm=1.0,
    weight_decay=0.0,
    gradient_checkpointing=True,
    bf16=(dtype == torch.bfloat16),
    fp16=(dtype == torch.float16),
    logging_steps=1,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)


## Demonstration: Execute a Training Run with PEFT

> Perform a training run while tracking metrics

In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    start_mem = torch.cuda.memory_allocated() / (1024 ** 3)
else:
    start_mem = 0.0

train_start = time.perf_counter()
train_result = trainer.train()
train_seconds = time.perf_counter() - train_start

In [ ]:

train_metrics = dict(train_result.metrics)
train_metrics["train_wall_seconds"] = float(train_seconds)
if torch.cuda.is_available():
    peak_mem = torch.cuda.max_memory_allocated() / (1024 ** 3)
    end_mem = torch.cuda.memory_allocated() / (1024 ** 3)
else:
    peak_mem = 0.0
    end_mem = 0.0

print("\nTraining metrics summary:")
for k, v in train_metrics.items():
    print(f"{k}: {v}")
print(f"start_mem_gb: {start_mem:.2f}")
print(f"peak_mem_gb: {peak_mem:.2f}")
print(f"end_mem_gb: {end_mem:.2f}")

log_history = trainer.state.log_history
train_steps = [x.get("step") for x in log_history if "loss" in x and "eval_loss" not in x]
train_losses = [x.get("loss") for x in log_history if "loss" in x and "eval_loss" not in x]
eval_steps = [x.get("step") for x in log_history if "eval_loss" in x]
eval_losses = [x.get("eval_loss") for x in log_history if "eval_loss" in x]

if train_steps:
    plt.figure(figsize=(9, 5))
    plt.plot(train_steps, train_losses, label="train_loss")
    if eval_steps:
        plt.plot(eval_steps, eval_losses, label="eval_loss")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.title("SFT Loss Curves")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
else:
    print("No log points available. Skipping loss chart.")


In [ ]:

final_dir = output_dir / "adapters"
model.save_pretrained(str(final_dir))

print(f"Saved adapters to: {final_dir}")
weights_file = final_dir / "adapter_model.safetensors"
if weights_file.exists():
    weights_size_mb = weights_file.stat().st_size / (1024 ** 2)
    print(f"Adapter weights file size: {weights_size_mb:.2f} MB")


In [ ]:
def parse_json_from_output(text: str) -> Dict[str, Any]:
    cleaned = text.strip()
    if cleaned.startswith("```"):
        lines = [line for line in cleaned.splitlines() if not line.strip().startswith("```")]
        cleaned = "\n".join(lines).strip()
    try:
        return json.loads(cleaned)
    except Exception:
        start = cleaned.find("{")
        end = cleaned.rfind("}")
        if start == -1 or end == -1 or end <= start:
            raise ValueError("No JSON object found in model output.")
        return json.loads(cleaned[start : end + 1])

def generate_structured_prediction(
    model_obj: AutoModelForCausalLM,
    input_obj: Dict[str, str],
    max_new_tokens: int = 220,
) -> Tuple[Optional[Dict[str, Any]], str]:
    prompt_text = input_obj.get("prompt_text")
    if prompt_text is None:
        infer_messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps(input_obj, ensure_ascii=False)},
        ]
        prompt_text = tokenizer.apply_chat_template(
            infer_messages, tokenize=False, add_generation_prompt=True
        )

    inputs = tokenizer(prompt_text, return_tensors="pt").to(model_obj.device)
    with torch.no_grad():
        output = model_obj.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output[0][inputs["input_ids"].shape[1] :]
    raw_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    try:
        parsed = parse_json_from_output(raw_text)
        normalized = {k: str(parsed.get(k, "")).strip() for k in REQUIRED_FIELDS}
        return normalized, raw_text
    except Exception:
        return None, raw_text

def evaluate_model(model_obj: AutoModelForCausalLM, split_ds: Dataset, label: str) -> Dict[str, Any]:
    sample_n = min(EVAL_SAMPLE_SIZE, len(split_ds))
    subset = split_ds.select(range(sample_n))
    rows = []
    started = time.perf_counter()

    print(f"Evaluating {label} model on {sample_n} samples...")
    for i, ex in enumerate(subset, start=1):
        pred_obj, raw = generate_structured_prediction(model_obj, ex["input_obj"] )
        gold = ex["target_obj"]

        schema_valid = 1.0
        if pred_obj is None:
            schema_valid = 0.0
            category_match = 0.0
            sub_category_match = 0.0
            tone_match = 0.0
        else:
            if any(not str(pred_obj.get(k, "")).strip() for k in REQUIRED_FIELDS):
                schema_valid = 0.0
            if not str(pred_obj.get("customer_id", "")).startswith("CUST-"):
                schema_valid = 0.0
            if str(pred_obj.get("tone_urgency", "")).strip().lower() not in {"low", "moderate", "high", "urgent"}:
                schema_valid = 0.0

            category_match = 1.0 if str(pred_obj.get("category", "")).strip() == str(gold.get("category", "")).strip() else 0.0
            sub_category_match = 1.0 if str(pred_obj.get("sub_category", "")).strip() == str(gold.get("sub_category", "")).strip() else 0.0
            tone_match = 1.0 if str(pred_obj.get("tone_urgency", "")).strip() == str(gold.get("tone_urgency", "")).strip() else 0.0

        weighted_score = 0.6 * schema_valid + 0.3 * category_match + 0.1 * tone_match
        rows.append(
            {
                "schema_valid": schema_valid,
                "category_match": category_match,
                "sub_category_match": sub_category_match,
                "tone_match": tone_match,
                "weighted_score": weighted_score,
                "raw_preview": raw[:220].replace("\n", " "),
            }
        )

        if i % 25 == 0 or i == sample_n:
            print(f"  Processed {i}/{sample_n} samples")

    elapsed = time.perf_counter() - started
    summary = {
        "label": label,
        "samples": sample_n,
        "seconds": elapsed,
        "samples_per_second": sample_n / elapsed if elapsed > 0 else 0.0,
        "schema_valid_rate": statistics.mean(r["schema_valid"] for r in rows),
        "category_accuracy": statistics.mean(r["category_match"] for r in rows),
        "sub_category_accuracy": statistics.mean(r["sub_category_match"] for r in rows),
        "tone_accuracy": statistics.mean(r["tone_match"] for r in rows),
        "weighted_score": statistics.mean(r["weighted_score"] for r in rows),
        "rows": rows,
    }
    return summary

In [ ]:
import gc
from peft import AutoPeftModelForCausalLM

# Release training resources before post-training evaluation.
if torch.cuda.is_available():
    del trainer
    del model
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()


# Load the model back for evaluation
model = AutoPeftModelForCausalLM.from_pretrained(
    str(final_dir),
    dtype=dtype if torch.cuda.is_available() else torch.float32,
    use_safetensors=True,
    device_map=device_map,
)


post_metrics = evaluate_model(model, eval_ds, label="lora_sft")
print("\nPost-training evaluation summary:")
for k, v in post_metrics.items():
    if k != "rows":
        print(f"{k}: {v}")